# ZA-GAS Model — Precipitation Analysis

**Model:** Zero-Augmented GAS(L,L) following Creal, Koopman & Lucas (2012).

**Observation model:**
$$p(y_t \mid \mathcal{F}_{t-1}) = (1-\pi_t)\,\mathbf{1}[y_t=0] + \pi_t\,g(y_t; f_t, \theta)\,\mathbf{1}[y_t > 0]$$

**Distribution:** GB2 with log-link parameterisation (`distributions/gb2_log_link.py`)

**GAS update:** `f_{t+1} = ω + Σ_{l∈L} A_l s_{t-l+1} + Σ_{l∈L} B_l f_{t-l+1}` (`models/gas_filter.py`)

**Pi dynamics:** AR-logistic with seasonal y-lags (`pi_dynamics/ar_logistic.py`)

---

In [ ]:
# ============================================================
# 0.  Imports and sys.path setup
# ============================================================
import sys
import os

# Add the FurtherTopics directory to sys.path so that the
# sub-packages (distributions, models, pi_dynamics, ...) are importable.
REPO_ROOT = os.path.dirname(os.path.abspath('.'))
FURTHER   = os.path.join(REPO_ROOT, 'FurtherTopics')
if FURTHER not in sys.path:
    sys.path.insert(0, FURTHER)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# --- Framework packages ---
from distributions.gb2_log_link import GB2LogLink
from pi_dynamics.factory        import PiDynamicsFactory
from models.za_gas_model        import ZAGASModel
from models.lags                import SEASONAL_LAGS
import diagnostics
from diagnostics import (
    pit_values, quantile_residuals,
    info_table, coverage_tests,
    jarque_bera,
    latex_info_table, latex_coverage_table,
    latex_jb_table, latex_simulation_table,
    plots,
)
from simulation.simulator import simulate_oos, evaluate_metrics

print('All packages loaded successfully.')

## 1. Data loading

Set `N_LOCATIONS = None` to process every station in the folders.

In [ ]:
# ============================================================
# 1a.  Directories and limits
# ============================================================
DAILY_DIR   = Path(r'C:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\data\output')
MONTHLY_DIR = Path(r'C:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\data\monthly')

# Limit the number of locations during development; set to None for all.
N_LOCATIONS = 3

In [ ]:
# ============================================================
# 1b.  Generic loader — returns {location: {y_train, y_test, ...}}
# ============================================================
def load_pairs(directory: Path, y_col: str, date_col: str, n_max=None) -> dict:
    """
    Scan `directory` for *_train.csv / *_test.csv pairs and return a dict
    keyed by location name.

    Parameters
    ----------
    directory : folder containing the CSV files
    y_col     : column name with the target variable
    date_col  : column name with the date / period label
    n_max     : if set, only load the first n_max pairs (sorted alphabetically)
    """
    train_files = sorted(
        f for f in directory.iterdir() if f.name.endswith('_train.csv')
    )
    if n_max:
        train_files = train_files[:n_max]

    data = {}
    for tf in train_files:
        loc      = tf.stem.replace('_train', '')
        test_f   = tf.parent / (loc + '_test.csv')
        if not test_f.exists():
            print(f'WARNING: no test file found for {loc}, skipping.')
            continue

        df_train = pd.read_csv(tf)
        df_test  = pd.read_csv(test_f)

        data[loc] = {
            'y_train'    : df_train[y_col].values.astype(float),
            'y_test'     : df_test[y_col].values.astype(float),
            'date_train' : df_train[date_col].values if date_col in df_train.columns else None,
            'date_test'  : df_test[date_col].values  if date_col in df_test.columns  else None,
        }
    return data


# Daily data: each row is one day
daily_data = load_pairs(
    DAILY_DIR,
    y_col    = 'PRECIPITACAO TOTAL, DIARIO(mm)',
    date_col = 'Data Medicao',
    n_max    = N_LOCATIONS,
)

# Monthly data: each row is one month (summed daily precipitation)
monthly_data = load_pairs(
    MONTHLY_DIR,
    y_col    = 'PRECIPITACAO TOTAL, DIARIO(mm)',
    date_col = 'year-month',
    n_max    = N_LOCATIONS,
)

print(f'Daily locations  : {list(daily_data.keys())}')
print(f'Monthly locations: {list(monthly_data.keys())}')

## 2. Model setup

The same `ZAGASModel` class is used for both cadences; only the `seasonal` argument changes, which determines the lag set **L**.

| Cadence  | Lag set L                     | Max lag |
|----------|-------------------------------|---------|
| daily    | [1, 2, 3, 364, 365, 366, 367] | 367     |
| monthly  | [1, 2, 3, 11, 12, 13]         | 13      |

To swap the distribution, replace `GB2LogLink()` with any other `Distribution` subclass.  
To swap the pi dynamics, call `PiDynamicsFactory.get('your_name')`.

In [ ]:
# ============================================================
# 2.  Model factory
# ============================================================
def build_model(seasonal: str) -> ZAGASModel:
    """
    Build a ZA-GAS model for the given cadence.

    Swap components here to test alternative specifications:
      - Distribution : replace GB2LogLink() with another Distribution subclass
      - Pi dynamics  : change 'ar_logistic' to another registered name
    """
    dist   = GB2LogLink()
    pi_dyn = PiDynamicsFactory.get('ar_logistic')

    return ZAGASModel(
        distribution = dist,
        pi_dynamics  = pi_dyn,
        seasonal     = seasonal,   # sets the lag set L
        scale_score  = True,       # inverse-Fisher scaling (Creal et al. type 1)
    )


# Show parameter counts
for seas in ('daily', 'monthly'):
    m = build_model(seas)
    n_gas = m.gas.codec.n_params
    n_pi  = len(m._pi_names)
    print(f'{seas:8s}  lag set={SEASONAL_LAGS[seas]}  '
          f'GAS params={n_gas}  pi params={n_pi}  total={n_gas+n_pi}')

## 3. Estimation

In [ ]:
# ============================================================
# 3a.  Fit daily models
# ============================================================
daily_results = {}

for loc, d in daily_data.items():
    print(f'\nFitting daily model — {loc}  (n_train={len(d["y_train"]):,})')
    model = build_model('daily')
    fit   = model.fit(d['y_train'], verbose=False)

    daily_results[loc] = {
        'model'      : model,
        'fit'        : fit,
        'y_train'    : d['y_train'],
        'y_test'     : d['y_test'],
        'date_train' : d['date_train'],
        'date_test'  : d['date_test'],
    }
    print(f'  loglik={fit["loglik"]:.2f}  |  success={fit["success"]}')

In [ ]:
# ============================================================
# 3b.  Fit monthly models
# ============================================================
monthly_results = {}

for loc, d in monthly_data.items():
    print(f'\nFitting monthly model — {loc}  (n_train={len(d["y_train"]):,})')
    model = build_model('monthly')
    fit   = model.fit(d['y_train'], verbose=False)

    monthly_results[loc] = {
        'model'      : model,
        'fit'        : fit,
        'y_train'    : d['y_train'],
        'y_test'     : d['y_test'],
        'date_train' : d['date_train'],
        'date_test'  : d['date_test'],
    }
    print(f'  loglik={fit["loglik"]:.2f}  |  success={fit["success"]}')

## 4. In-sample diagnostics

In [ ]:
# ============================================================
# 4a.  Compute in-sample filtered paths + all diagnostic objects
# ============================================================
def compute_is_diagnostics(results: dict, alphas=(0.05, 0.10, 0.25)) -> dict:
    """
    For each fitted location, compute:
      - filtered paths  (phi_t, xi_t, pi_t)
      - PIT values and quantile residuals
      - AIC / BIC
      - Kupiec + Christoffersen coverage tests at several levels
      - Jarque-Bera normality test on quantile residuals
    """
    diag = {}
    for loc, r in results.items():
        model = r['model']
        theta = r['fit']['theta']
        y     = r['y_train']

        # Total number of free parameters
        n_pars = model.gas.codec.n_params + len(model._pi_names)

        # Filtered paths
        paths  = model.filter(theta, y)
        n_eff  = len(paths['y_eff'])

        # CDF values and derived quantities
        cdfs   = model.cdf_series(theta, y)
        pit    = pit_values(cdfs, paths['y_eff'], randomise_zeros=True)
        qr     = quantile_residuals(cdfs, paths['y_eff'], randomise_zeros=True)

        # Information criteria
        ic     = info_table(paths['loglik'], n_pars, n_eff)

        # Coverage tests
        cov    = coverage_tests(pit, alphas=list(alphas))

        # Jarque-Bera
        jb     = jarque_bera(qr)

        diag[loc] = dict(
            paths=paths, cdfs=cdfs, pit=pit, qr=qr,
            ic=ic, coverage=cov, jb=jb,
        )
    return diag


daily_is   = compute_is_diagnostics(daily_results)
monthly_is = compute_is_diagnostics(monthly_results)
print('In-sample diagnostics computed.')

In [ ]:
# ============================================================
# 4b.  Information criteria — printed and LaTeX
# ============================================================
def print_ic(label, diag):
    print(f'\n=== {label} — Information Criteria ===')
    print(f'{"Location":<30} {"k":>5} {"n":>6} {"loglik":>10} {"AIC":>10} {"BIC":>10}')
    print('-' * 70)
    for loc, d in diag.items():
        ic = d['ic']
        print(f'{loc:<30} {ic["n_params"]:>5} {ic["n_obs"]:>6} '
              f'{ic["loglik"]:>10.2f} {ic["aic"]:>10.2f} {ic["bic"]:>10.2f}')

print_ic('DAILY',   daily_is)
print_ic('MONTHLY', monthly_is)

In [ ]:
# LaTeX version
print(latex_info_table(
    {loc: d['ic'] for loc, d in daily_is.items()},
    caption='Daily ZA-GAS — information criteria',
    label='tab:daily_ic',
))

In [ ]:
# ============================================================
# 4c.  Coverage tests
# ============================================================
def print_coverage(label, diag):
    print(f'\n=== {label} — Coverage Tests ===')
    for loc, d in diag.items():
        print(f'\n  {loc}')
        print(f'  {"alpha":<8} {"viol.":<8} {"rate":<8} {"p_uc":<10} {"p_cc":<10} reject_cc')
        for row in d['coverage']:
            print(f'  {row["alpha"]:<8.2f} {row["violations"]:<8} '
                  f'{row["violation_rate"]:<8.3f} {row["pvalue_uc"]:<10.4f} '
                  f'{row["pvalue_cc"]:<10.4f} {row["reject_cc"]}')

print_coverage('DAILY',   daily_is)
print_coverage('MONTHLY', monthly_is)

In [ ]:
# LaTeX coverage table
print(latex_coverage_table(
    {loc: d['coverage'] for loc, d in daily_is.items()},
    caption='Daily ZA-GAS — coverage tests (in-sample)',
    label='tab:daily_coverage',
))

In [ ]:
# ============================================================
# 4d.  Jarque-Bera on in-sample quantile residuals
# ============================================================
def print_jb(label, diag):
    print(f'\n=== {label} — Jarque-Bera ===')
    print(f'{"Location":<30} {"Skew":<10} {"Kurt":<10} {"JB":>8} {"p-val":>10}')
    for loc, d in diag.items():
        jb = d['jb']
        print(f'{loc:<30} {jb["skewness"]:<10.3f} {jb["kurtosis"]:<10.3f} '
              f'{jb["stat"]:>8.2f} {jb["pvalue"]:>10.4f}')

print_jb('DAILY',   daily_is)
print_jb('MONTHLY', monthly_is)

print()
print(latex_jb_table(
    {loc: d['jb'] for loc, d in daily_is.items()},
    caption='Daily ZA-GAS — Jarque-Bera on quantile residuals',
    label='tab:daily_jb',
))

## 5. Diagnostic plots

When multiple locations are loaded the helper `plots.mosaic()` arranges one panel per location in a grid.

In [ ]:
# ============================================================
# 5a.  Quantile-residual mosaic — daily
# ============================================================
datasets_daily_qr = {
    loc: dict(
        residuals = d['qr'],
        pit       = d['pit'],
        dates     = daily_results[loc]['date_train'],
    )
    for loc, d in daily_is.items()
}

if len(datasets_daily_qr) == 1:
    loc = list(datasets_daily_qr)[0]
    fig = plots.quantile_residual_panel(**datasets_daily_qr[loc], location=loc)
else:
    fig = plots.mosaic(plots.quantile_residual_panel, datasets_daily_qr, ncols=2)
    fig.suptitle('Daily ZA-GAS — quantile residual diagnostics', fontsize=14)

plt.savefig('daily_qr_mosaic.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 5b.  Filtered state paths — first daily location
# ============================================================
first_loc   = list(daily_results.keys())[0]
first_paths = daily_is[first_loc]['paths']

fig = plots.filtered_paths_plot(
    first_paths,
    dates    = daily_results[first_loc]['date_train'],
    location = first_loc,
)
plt.savefig(f'filtered_paths_{first_loc}.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 5c.  Monthly quantile-residual mosaic
# ============================================================
datasets_monthly_qr = {
    loc: dict(
        residuals = d['qr'],
        pit       = d['pit'],
        dates     = monthly_results[loc]['date_train'],
    )
    for loc, d in monthly_is.items()
}

if len(datasets_monthly_qr) == 1:
    loc = list(datasets_monthly_qr)[0]
    fig = plots.quantile_residual_panel(**datasets_monthly_qr[loc], location=loc)
else:
    fig = plots.mosaic(plots.quantile_residual_panel, datasets_monthly_qr, ncols=2)
    fig.suptitle('Monthly ZA-GAS — quantile residual diagnostics', fontsize=14)

plt.savefig('monthly_qr_mosaic.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Out-of-sample evaluation

`evaluate_metrics` runs a 1-step-ahead rolling evaluation on the test set, starting from the terminal training state.

In [ ]:
# ============================================================
# 6a.  Daily OOS metrics
# ============================================================
daily_metrics = {}

for loc, r in daily_results.items():
    print(f'Evaluating OOS — {loc}')
    m = evaluate_metrics(
        model   = r['model'],
        theta   = r['fit']['theta'],
        y_train = r['y_train'],
        y_test  = r['y_test'],
        n_draws = 300,    # MC draws per step for CRPS
        seed    = 42,
    )
    daily_metrics[loc] = m
    print(f'  IS   RMSE={m["is_rmse"]:.3f}  MAD={m["is_mad"]:.3f}  CRPS={m["is_crps"]:.3f}')
    print(f'  OOS  RMSE={m["oos_rmse"]:.3f}  MAD={m["oos_mad"]:.3f}  CRPS={m["oos_crps"]:.3f}')

In [ ]:
# ============================================================
# 6b.  Monthly OOS metrics
# ============================================================
monthly_metrics = {}

for loc, r in monthly_results.items():
    print(f'Evaluating OOS — {loc}')
    m = evaluate_metrics(
        model   = r['model'],
        theta   = r['fit']['theta'],
        y_train = r['y_train'],
        y_test  = r['y_test'],
        n_draws = 300,
        seed    = 42,
    )
    monthly_metrics[loc] = m
    print(f'  IS   RMSE={m["is_rmse"]:.3f}  MAD={m["is_mad"]:.3f}  CRPS={m["is_crps"]:.3f}')
    print(f'  OOS  RMSE={m["oos_rmse"]:.3f}  MAD={m["oos_mad"]:.3f}  CRPS={m["oos_crps"]:.3f}')

In [ ]:
# ============================================================
# 6c.  Summary metric tables — printed + LaTeX
# ============================================================
def build_sim_dict(metrics):
    return {
        loc: {
            'rmse_is' : m['is_rmse'],  'mad_is' : m['is_mad'],  'crps_is': m['is_crps'],
            'rmse'    : m['oos_rmse'], 'mad'    : m['oos_mad'], 'crps'   : m['oos_crps'],
        }
        for loc, m in metrics.items()
    }

print(latex_simulation_table(
    build_sim_dict(daily_metrics),
    caption='Daily ZA-GAS — forecast evaluation',
    label='tab:daily_sim',
))
print()
print(latex_simulation_table(
    build_sim_dict(monthly_metrics),
    caption='Monthly ZA-GAS — forecast evaluation',
    label='tab:monthly_sim',
))

In [ ]:
# ============================================================
# 6d.  OOS Jarque-Bera and coverage tests
# ============================================================
print('=== DAILY OOS — Jarque-Bera ===')
print(f'{"Location":<30} {"Skew":<10} {"Kurt":<10} {"JB":>8} {"p-val":>10}')
for loc, m in daily_metrics.items():
    jb = m['oos_jb']
    print(f'{loc:<30} {jb["skewness"]:<10.3f} {jb["kurtosis"]:<10.3f} '
          f'{jb["stat"]:>8.2f} {jb["pvalue"]:>10.4f}')

print()
print(latex_jb_table(
    {loc: m['oos_jb'] for loc, m in daily_metrics.items()},
    caption='Daily ZA-GAS — OOS Jarque-Bera on quantile residuals',
    label='tab:daily_oos_jb',
))

# OOS coverage tests
oos_coverage = {
    loc: coverage_tests(m['oos_pit'], alphas=[0.05, 0.10, 0.25])
    for loc, m in daily_metrics.items()
}
print()
print(latex_coverage_table(
    oos_coverage,
    caption='Daily ZA-GAS — OOS coverage tests',
    label='tab:daily_oos_coverage',
))

In [ ]:
# ============================================================
# 6e.  OOS quantile-residual mosaic — daily
# ============================================================
oos_qr_ds = {
    loc: dict(
        residuals = m['oos_qr'],
        pit       = m['oos_pit'],
        dates     = daily_results[loc]['date_test'],
    )
    for loc, m in daily_metrics.items()
}

if len(oos_qr_ds) == 1:
    loc = list(oos_qr_ds)[0]
    fig = plots.quantile_residual_panel(**oos_qr_ds[loc], location=f'{loc} (OOS)')
else:
    fig = plots.mosaic(plots.quantile_residual_panel, oos_qr_ds, ncols=2)
    fig.suptitle('Daily ZA-GAS OOS — quantile residual diagnostics', fontsize=14)

plt.savefig('daily_oos_qr_mosaic.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Save results

In [ ]:
# ============================================================
# 7.  Persist estimated parameters and metrics to JSON
# ============================================================
import json
import numpy as np

def _clean(obj):
    """JSON serialiser for numpy types."""
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.bool_):    return bool(obj)
    return str(obj)

save = {}
for label, results, metrics_d in [
    ('daily',   daily_results,   daily_metrics),
    ('monthly', monthly_results, monthly_metrics),
]:
    save[label] = {}
    for loc, r in results.items():
        m = metrics_d.get(loc, {})
        save[label][loc] = {
            'loglik'   : float(r['fit']['loglik']),
            'success'  : bool(r['fit']['success']),
            'theta'    : r['fit']['theta'].tolist(),
            'is_rmse'  : float(m.get('is_rmse',  float('nan'))),
            'is_mad'   : float(m.get('is_mad',   float('nan'))),
            'is_crps'  : float(m.get('is_crps',  float('nan'))),
            'oos_rmse' : float(m.get('oos_rmse', float('nan'))),
            'oos_mad'  : float(m.get('oos_mad',  float('nan'))),
            'oos_crps' : float(m.get('oos_crps', float('nan'))),
        }

with open('results_za_gas.json', 'w') as f:
    json.dump(save, f, indent=2, default=_clean)

print('Results saved to results_za_gas.json')